## Elhogym Beta Testing

We'll start by importing the required modules.

In [1]:
# modules
import os
import gym
import numpy as np

# agent
from agent.sac.sac import SACAgent
from agent.replay_buffer import ReplayBuffer
from agent.cfgs.parser import parse_cfg

# env
sys.path.insert(0, '/workspaces/elizabeth-homes/exp/hannes')
import input
from sinergym.utils.rewards import *
from sinergym.utils.wrappers import NormalizeObservation
env = gym.make("Eplus-1storeytest-v2")

# normalise observation space using sinergym wrapper
ranges = {}
for feat in env.variables['observation']:
    ranges[feat] = [-1, 1]
env = NormalizeObservation(env, ranges=ranges)

/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:73: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gym/envs/registration.py:216: UserWarning: WARN: Overriding environment Eplus-demo-v1
  logger.warn("Overriding environment {}".format(id))
no file found at given path, content will be considered as empty (GBR_ENG_London.Wea.Ctr-St.James.Park.037700_TMYx.2004-2018.rain)
[2022-09-30 14:14:25,316] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf ExternalInterface object if it is not present...
[2022-09-30 14:14:25,320] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf Site:Location and SizingPeriod:DesignDay(s) to weather and ddy file...
[2022-09-30 14:14:25,362] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Updating idf OutPut:Variable and variables XML tree model for BVCTB connection.
[2022-09-30 14:14:25,371] EPLUS_ENV_1storeytest-v2_MainThread_ROOT INFO:Setting up extra configuratio

First we'll configure the agent to interface with the Elhogym environment.

In [4]:
[env.action_space.low[0], env.action_space.low[0]] 

[-1.0, -1.0]

In [2]:
cfg = parse_cfg()
cfg.sac.obs_dim = env.observation_space.shape[0]
cfg.sac.action_dim = env.action_space.shape[0]
cfg.sac.action_range = [env.action_space.low[0], env.action_space.high[0]]

UnsupportedValueType: Value 'float32' is not a supported primitive type
    full_key: action_range[0]
    object_type=list

Next we'll setup a function to `evaluate` the agent performance at set intervals. We'll use cumulative reward over the first month of environment interaction as the criterion.

In [4]:
def evaluate(agent, cfg):
    eval_env = gym.make("Eplus-1storeytest-v2")
    
    eval_obs = eval_env.reset()
    eval_rewards = 0

    for j in range(cfg.eval_steps):
        with utils.eval_mode(agent):
            action = agent.act(eval_obs, sample=False)        
        eval_obs, reward, _, _ = eval_env.step(action)
        eval_rewards += reward

    return eval_rewards

In [3]:
env.action_space

Box([-1.], [1.], (1,), float32)